In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import torch, sys
sys.path.insert(0, "/content/drive/MyDrive/sn-grok")
!pip install transformer_lens
from sngrok.model import SnMLP

run_dir = "/content/drive/MyDrive/sn-grok/checkpoints/s4_new_50_0"
ckpt = torch.load(f"{run_dir}/80000.pth", map_location="cpu")

print(ckpt.keys())          # dict_keys(['model', 'optimizer', 'config', 'rng'])
print(ckpt["config"])       # {'vocab_size': 24, 'embed_dim': 256, 'model_dim': 128, ...}

for name, tensor in ckpt["model"].items():
    print(name, tuple(tensor.shape), tensor.dtype)
# lembed.weight (24, 256); rembed.weight (24, 256);
# linear.weight (128, 512); unembed.weight (24, 128); unembed.bias (24)

model = SnMLP.from_config(ckpt["config"])
model.load_state_dict(ckpt["model"])
model.eval()

dict_keys(['model', 'optimizer', 'config', 'rng'])
{'embed_dim': 256, 'model_dim': 128, 'vocab_size': 24, 'tie_lr': False}
lembed.weight (24, 256) torch.float32
rembed.weight (24, 256) torch.float32
linear.weight (128, 512) torch.float32
unembed.weight (24, 128) torch.float32
unembed.bias (24,) torch.float32


SnMLP(
  (lembed): Embedding(24, 256)
  (rembed): Embedding(24, 256)
  (linear): Linear(in_features=512, out_features=128, bias=False)
  (unembed): Linear(in_features=128, out_features=24, bias=True)
  (hook_lembed): HookPoint(name='hook_lembed')
  (hook_rembed): HookPoint(name='hook_rembed')
  (hook_linear): HookPoint(name='hook_linear')
  (hook_unembed): HookPoint(name='hook_unembed')
)

In [ ]:
from pathlib import Path
import polars as pl
run_dir = Path(run_dir)
run_data = pl.read_parquet(run_dir/'data.parquet')
l = torch.as_tensor(run_data['index_left'].to_numpy())
r = torch.as_tensor(run_data['index_right'].to_numpy())
y = torch.as_tensor(run_data['index_target'].to_numpy())
in_train = torch.as_tensor(run_data['in_train'].to_numpy()) == 1

with torch.no_grad():                      # eval() doesn't disable autograd by itself
    logits = model(l, r)                   # [576, 24]
pred = logits.argmax(-1)

print('train acc', (pred[in_train]  == y[in_train]).float().mean().item())
print('test  acc', (pred[~in_train] == y[~in_train]).float().mean().item())

train acc 1.0
test  acc 1.0


In [ ]:
import sys
sys.path.insert(0, '/content/sn-grok')        # repo root, so `calculate_cosets` imports
import polars as pl
from sngrok.proper_subgroups_s4 import all_s4_subgroups
from calculate_cosets import make_full_coset_df

left_coset_df, right_coset_df = make_full_coset_df(all_s4_subgroups, 4)

print(left_coset_df.shape)                    # expect (576*28, 3) = (16128, 3)
print(left_coset_df.head())

(672, 3)
shape: (5, 3)
┌──────────────┬───────────────────────────┬──────────────┐
│ permutation  ┆ subgroup                  ┆ coset_rep    │
│ ---          ┆ ---                       ┆ ---          │
│ str          ┆ str                       ┆ str          │
╞══════════════╪═══════════════════════════╪══════════════╡
│ (2, 3, 0, 1) ┆ C2_single_transposition_0 ┆ (2, 3, 0, 1) │
│ (2, 3, 1, 0) ┆ C2_single_transposition_0 ┆ (2, 3, 0, 1) │
│ (0, 2, 3, 1) ┆ C2_single_transposition_0 ┆ (0, 2, 3, 1) │
│ (1, 2, 3, 0) ┆ C2_single_transposition_0 ┆ (0, 2, 3, 1) │
│ (0, 2, 1, 3) ┆ C2_single_transposition_0 ┆ (0, 2, 1, 3) │
└──────────────┴───────────────────────────┴──────────────┘


In [ ]:
from sngrok.groups import Symmetric
from sngrok.fourier import slow_sn_ft_1d, sn_fourier_basis
from calculate_cosets import fourier_basis_to_df

S4 = Symmetric(4)
W = model.linear.weight
embed_dim = model.embed_dim

left_ft  = slow_sn_ft_1d(model.lembed.weight @ W[:, :embed_dim].T, 4)
right_ft = slow_sn_ft_1d(model.rembed.weight @ W[:, embed_dim:].T, 4)

left_df  = fourier_basis_to_df(sn_fourier_basis(left_ft,  S4), 4, 'left_linear')
right_df = fourier_basis_to_df(sn_fourier_basis(right_ft, S4), 4, 'right_linear')

In [ ]:
from calculate_cosets import _all_data_coset_analysis

left_scores  = _all_data_coset_analysis(left_df,  right_coset_df)   # left acts ↔ right cosets
right_scores = _all_data_coset_analysis(right_df, left_coset_df)

print(left_scores.sort('coset_var_ratio').head(10))
print(right_scores.sort('coset_var_ratio').head(10))

shape: (10, 11)
┌────────────┬──────────┬───────────┬───────────┬───┬──────────┬───────────┬───────────┬───────────┐
│ layer      ┆ variable ┆ subgroup  ┆ min_coset ┆ … ┆ full_var ┆ two_norm  ┆ coset_var ┆ subgroup_ │
│ ---        ┆ ---      ┆ ---       ┆ _var      ┆   ┆ ---      ┆ ---       ┆ _ratio    ┆ class     │
│ str        ┆ str      ┆ str       ┆ ---       ┆   ┆ f32      ┆ f32       ┆ ---       ┆ ---       │
│            ┆          ┆           ┆ f32       ┆   ┆          ┆           ┆ f32       ┆ str       │
╞════════════╪══════════╪═══════════╪═══════════╪═══╪══════════╪═══════════╪═══════════╪═══════════╡
│ left_linea ┆ dim47    ┆ D8_1      ┆ 5.8297e-8 ┆ … ┆ 1.912619 ┆ 43.990234 ┆ 3.0480e-8 ┆ D8        │
│ r          ┆          ┆           ┆           ┆   ┆          ┆           ┆           ┆           │
│ left_linea ┆ dim103   ┆ D8_1      ┆ 7.3425e-8 ┆ … ┆ 1.517362 ┆ 34.899334 ┆ 4.8390e-8 ┆ D8        │
│ r          ┆          ┆           ┆           ┆   ┆          ┆           

In [ ]:
!ls /content/sn-grok/sngrok

cosets.py	    fourier.py	 permutations.py	  toy_code.py
cyclic_irreps.py    groups.py	 product_permutations.py  utils.py
data		    __init__.py  proper_subgroups.py
dihedral_irreps.py  irreps.py	 proper_subgroups_s4.py
dihedral.py	    model	 tableau.py
